In [15]:
from dataclasses import dataclass
import dask
import dask.dataframe as dd
from dask.distributed import Client, progress
from rich.console import Console
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
sys.path.append('~/projects/AbstractForecast/config')
from config import config

console = Console(highlight=False)
client = Client(n_workers=4, threads_per_worker=8)

In [16]:
def pshape(df):
    print(f'{df.shape[0]:,} rows')

In [17]:
data_path = config.data.staged / 'psychology'
if data_path.exists():
    console.print(f'[green] Found data path [/green]')
else:
    console.print(f'[red] Failed to find data path [/red]')

 Found data path 


In [25]:
df = dd.read_parquet(data_path, engine='fastparquet')
df = df.persist()
progress(df)
df = df.compute()
pshape(df)

[###############                         ] | 37% Completed |  0.1s

[###############                         ] | 37% Completed |  0.2s

[###############                         ] | 37% Completed |  0.3s

[###############                         ] | 37% Completed |  0.4s

[################                        ] | 40% Completed |  0.5s

...

[############################            ] | 70% Completed |  0.7s

[##############################          ] | 76% Completed |  0.8s

[######################################  ] | 95% Completed |  0.9s

[########################################] | 100% Completed |  1.0s

2,064,088 rows


In [26]:
df = df[df['language'] == 'en']
df = df[df['type'] == 'article']
pshape(df)

1,440,797 rows


In [27]:
df['abstract_len'] = df['abstract'].apply(lambda x : len(x))
plt.figure()
plt.hist(df['abstract_len'])
plt.show()
stats = df['abstract_len'].describe()
print(stats)

count    1.440797e+06
mean     1.119304e+03
std      9.922367e+02
min      1.000000e+00
25%      6.460000e+02
50%      9.890000e+02
75%      1.385000e+03
max      4.301600e+04


Name: abstract_len, dtype: float64


In [28]:
# Filter out long/short abstracts (+- 3 std)
mean = stats['mean']
std = stats['std']
limit = std * 3
high = mean + limit
low = max(mean - limit, 300)
df = df[(df['abstract_len'] > low) & (df['abstract_len'] < high)]
pshape(df)

1,285,394 rows


In [29]:
# Filter abstracts contianing 'exclude' strings, remove elements of strings fitting 'remove' regex patterns 
@dataclass(frozen=True)
class Filter:
    exclude = ['keywords:', 'Keywords:' 'query=', 'http', 'Abstract', 'Abstract '
    'ADVERTISEMENT RETURN TO ISSUE', 'PAPER ACCEPTED FOR PUBLICATION'
    'Article Views', 'Altimetric-Citations', 
    'Copyright', 'copyright', '©'
    'reference to this paper', 'Google Scholar'
    # foreign connectives (remember to include space)
    'de ',
    
    # chinese characters
   # Particles and function words
    '的', '了', '在', '是', '和', '与', '及', '或', '为', '被',
    '有', '无', '以', '对', '对于', '根据', '按', '由',
    
    # Common academic connectives
    '因此', '而且', '然而', '但', '但是', '同时', '并', '并且',
    '此外', '另外', '进一步', '总之', '综上', '可见',
    
    # Common verbs (often semantically weak in abstracts)
    '表明', '显示', '证明', '说明', '指出', '认为', '发现',
    '提出', '方法', '研究', '分析', '讨论', '介绍', 
     # Common single char words
    '的','了','是','在','和','与','对','有','无','以','为','被','等',
        # Vowels with accents
    'à', 'á', 'â', 'ã', 'ä', 'å', 'æ',
    'è', 'é', 'ê', 'ë',
    'ì', 'í', 'î', 'ï',
    'ò', 'ó', 'ô', 'õ', 'ö', 'ø', 'œ',
    'ù', 'ú', 'û', 'ü',
    'ý', 'ÿ',
    
    # Uppercase versions
    'À', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Æ',
    'È', 'É', 'Ê', 'Ë',
    'Ì', 'Í', 'Î', 'Ï',
    'Ò', 'Ó', 'Ô', 'Õ', 'Ö', 'Ø', 'Œ',
    'Ù', 'Ú', 'Û', 'Ü',
    'Ý',
    
    # Consonants with diacriticals
    'ç', 'Ç',
    'ñ', 'Ñ',
    'ð', 'Ð',
    'þ', 'Þ',
    'ß',

    ]
    tails = [
    'English', 'english',
    '<' , '>', ';', '@', '?', '[', ']', '{', '}'
    '#', '~', '/', '-', '_', '+', '=', '\\', '`', '¬', 
    '!', '£', '$', '%','^', '&', '*', '(', ')' 
    ]
    remove = []
@dataclass(frozen=True)
class Requirements:
    end_with = '.'

pshape(df)
filt = Filter()
df = df[~df['abstract'].str.contains('|'.join(filt.exclude))]
df = df[~df['abstract'].str.startswith('|'.join(filt.tails))]
df = df[~df['abstract'].str.endswith('|'.join(filt.tails))]

req = Requirements()
df = df[df['abstract'].str.endswith(req.end_with)]
pshape(df)

1,285,394 rows


1,030,975 rows


In [30]:
def sample(n, df, cols):
    mask = np.random.randint(0,df.shape[0]-1, (n,))
    samp = df.iloc[mask, :]
    return samp[cols].reset_index()
cols = ['abstract', 'abstract_len', 'cited_by_count', 'language']
samp = sample(2, df, cols)
for i in range(samp.shape[0]):
    print('\n======') 
    for c in cols:
        print(samp.loc[i, c])

The current study used a novel methodology based on multivocal ethnography to assess the relations between conformity and evaluations of intelligence and good behavior among Western (U.S.) and non‐Western (Ni‐Vanuatu) children (6‐ to 11‐year‐olds) and adolescents (13‐ to 17‐year‐olds; N = 256). Previous research has shown that U.S. adults were less likely to endorse high‐conformity children as intelligent than Ni‐Vanuatu adults. The current data demonstrate that in contrast to prior studies documenting cultural differences between adults' evaluations of conformity, children and adolescents in the United States and Vanuatu have a conformity bias when evaluating peers' intelligence and behavior. Conformity bias for good behavior increases with age. The results have implications for understanding the interplay of conformity bias and trait psychology across cultures and development.


891


25


en


“Kanata/Canada: Re-storying ‘Canada 150’ at the Canadian Museum for Human Rights” seeks to contextualize the changing role of museums and of heritage institutions within contemporary discussions about the urgent need for public education on Indigenous histories and contemporary realities. The author of this article argues that museums can become truly decolonizing spaces if they are willing to re-examine their own purpose and mandate. Through an examination of the CMHR’s own exhibition development for 2017, she maintains that undertaking grounded, reparative reconciliation that is meaningful to communities in a museum context means going beyond acknowledgement and recognition to re-storying the very foundations of Canadian nation-building, and of projects like Confederation that remain, necessarily, unfinished.


822


1


en


In [31]:
df_out = dd.from_pandas(df, npartitions=64)
df_out = dd.to_parquet(
    df_out,
    str(config.data.staged / 'psychology_clean'),
    engine = 'pyarrow',
    compression = 'zstd',
    compression_level = 1,
    write_statistics = True,
    compute = False,
    overwrite = True,
)
progress(client.compute(df_out))
print('done')

SyntaxError: unexpected character after line continuation character (<input_31>, line 10)